# ChemWorld Physics Sanity Check

This notebook probes whether `BatchReactorWorld` behaves like a plausible semi-mechanistic chemical world rather than an arbitrary black-box function. It keeps the hidden world seed fixed and scans one factor at a time.


## Kernel

Use the `Python (ChemWorld)` Jupyter kernel installed from the project `.venv`. If imports fail, run `python -m pip install -e ".[dev,notebooks]"` in the project root and reinstall the kernel with `python -m ipykernel install --user --name chemworld --display-name "Python (ChemWorld)"`.


In [1]:
import math
from pathlib import Path

import gymnasium as gym
import pandas as pd

import chemworld  # registers BatchReactorWorld

pd.set_option("display.precision", 4)
ROOT = Path.cwd() if (Path.cwd() / "pyproject.toml").exists() else Path.cwd().parent
print("chemworld version:", chemworld.__version__)
print("project root:", ROOT)


chemworld version: 0.1.0
project root: D:\Projects\ChemWorld


In [2]:
def scalar(observation, key: str) -> float | None:
    value = float(observation[key][0])
    return None if math.isnan(value) else value


def run_recipe(
    *,
    target_temperature_K: float = 390.0,
    duration_s: float = 300.0,
    catalyst: int = 1,
    solvent: int = 2,
    amount_mol: float = 0.010,
    seed: int = 31,
) -> dict[str, float | int | None]:
    actions = [
        {"operation": "add_solvent", "volume_L": 0.030, "solvent": solvent},
        {"operation": "add_reagent", "amount_mol": amount_mol},
        {"operation": "add_catalyst", "catalyst_amount_mol": 0.00025, "catalyst": catalyst},
        {
            "operation": "heat",
            "target_temperature_K": target_temperature_K,
            "duration_s": duration_s,
            "stirring_speed_rpm": 720.0,
        },
        {"operation": "terminate"},
        {"operation": "measure", "instrument": "final_assay"},
    ]
    env = gym.make(
        "BatchReactorWorld",
        world_split="public-dev",
        budget=len(actions),
        seed=seed,
    )
    env.reset(seed=seed)
    try:
        for action in actions:
            observation, _reward, _terminated, _truncated, info = env.step(action)
        return {
            "temperature_K": target_temperature_K,
            "duration_s": duration_s,
            "catalyst": catalyst,
            "solvent": solvent,
            "amount_mol": amount_mol,
            "yield": scalar(observation, "yield"),
            "selectivity": scalar(observation, "selectivity"),
            "conversion": scalar(observation, "conversion"),
            "degradation": scalar(observation, "degradation_warning"),
            "byproduct": scalar(observation, "byproduct_signal"),
            "risk": scalar(observation, "safety_risk"),
            "cost": scalar(observation, "cost"),
            "score": scalar(observation, "score"),
            "leaderboard_score": info["leaderboard_score"],
            "observed_keys": len(info["observed_keys"]),
        }
    finally:
        env.close()


## Temperature Scan

At a fixed short reaction time, moderate heating improves conversion, while excessive heating increases degradation and risk.


In [3]:
temperature_df = pd.DataFrame(
    run_recipe(target_temperature_K=temp, duration_s=300.0)
    for temp in [300.0, 330.0, 360.0, 390.0, 420.0, 450.0]
)
temperature_df[["temperature_K", "yield", "conversion", "degradation", "risk", "score"]]


,temperature_K,yield,conversion,degradation,risk,score
0,300.0,0.1060,0.1461,0.0067,0.0943,0.1750
1,330.0,0.2659,0.3538,0.0122,0.0973,0.2753
2,360.0,0.4545,0.6390,0.0416,0.1069,0.3836
3,390.0,0.5289,0.8650,0.1300,0.1645,0.4033
4,420.0,0.4400,0.9677,0.2856,0.3185,0.2910
5,450.0,0.2851,0.9966,0.4562,0.3931,0.1554


## Time Scan

Longer reaction time can initially help conversion, but product degradation eventually dominates.


In [4]:
time_df = pd.DataFrame(
    run_recipe(target_temperature_K=360.0, duration_s=time_s)
    for time_s in [60.0, 120.0, 300.0, 600.0, 900.0, 1800.0, 3600.0]
)
time_df[["duration_s", "yield", "conversion", "degradation", "risk", "score"]]


,duration_s,yield,conversion,degradation,risk,score
0,60.0,0.0565,0.0877,0.0064,0.1001,0.1290
1,120.0,0.1875,0.2570,0.0095,0.1058,0.2205
2,300.0,0.4545,0.6390,0.0416,0.1069,0.3836
3,600.0,0.5622,0.8865,0.1245,0.1068,0.4418
4,900.0,0.5336,0.9633,0.2109,0.1068,0.4145
5,1800.0,0.3541,1.0000,0.4186,0.1068,0.2768
6,3600.0,0.1387,1.0000,0.6357,0.1068,0.1129


## Catalyst-Solvent Interaction

The best condition is not a single globally dominant catalyst or solvent; interactions matter.


In [5]:
interaction_df = pd.DataFrame(
    run_recipe(catalyst=catalyst, solvent=solvent, target_temperature_K=360.0, duration_s=600.0)
    for catalyst in range(4)
    for solvent in range(4)
)
interaction_df.pivot(index="catalyst", columns="solvent", values="score")


solvent,0,1,2,3
catalyst,,,,
0,0.3467,0.3130,0.3253,0.2571
1,0.4723,0.4306,0.4418,0.3606
2,0.2615,0.2331,0.2450,0.1929
3,0.4092,0.3692,0.3821,0.3039


## Concentration-Risk Trade-off

Higher initial reagent loading increases the pressure/risk proxy and can reduce balanced objective score even when conversion remains high.


In [6]:
risk_df = pd.DataFrame(
    run_recipe(amount_mol=amount, target_temperature_K=390.0, duration_s=1800.0)
    for amount in [0.004, 0.008, 0.012, 0.016, 0.020]
)
risk_df[["amount_mol", "yield", "conversion", "risk", "score"]]


,amount_mol,yield,conversion,risk,score
0,0.004,0.0693,1.0,0.1515,0.0546
1,0.008,0.0691,1.0,0.1586,0.0508
2,0.012,0.0689,1.0,0.1704,0.0459
3,0.016,0.0687,1.0,0.1884,0.0395
4,0.020,0.0685,1.0,0.2132,0.0313


## Sanity Assertions

These checks are deliberately qualitative. They are meant to catch broken model behavior before benchmark results are trusted.


In [7]:
def pick(frame: pd.DataFrame, column: str, **selector: float) -> float:
    mask = pd.Series(True, index=frame.index)
    for key, value in selector.items():
        mask &= frame[key] == value
    return float(frame.loc[mask, column].iloc[0])


checks = {
    "moderate_heat_improves_conversion": pick(
        temperature_df, "conversion", temperature_K=390.0
    )
    > pick(temperature_df, "conversion", temperature_K=300.0),
    "excess_heat_increases_degradation": pick(
        temperature_df, "degradation", temperature_K=450.0
    )
    > pick(temperature_df, "degradation", temperature_K=390.0),
    "long_time_increases_degradation": pick(time_df, "degradation", duration_s=3600.0)
    > pick(time_df, "degradation", duration_s=300.0),
    "higher_loading_increases_risk": pick(risk_df, "risk", amount_mol=0.020)
    > pick(risk_df, "risk", amount_mol=0.004),
    "final_assay_populates_leaderboard_score": bool(
        risk_df["leaderboard_score"].notna().all()
    ),
}
pd.DataFrame([checks]).T.rename(columns={0: "passed"})


,passed
moderate_heat_improves_conversion,True
excess_heat_increases_degradation,True
long_time_increases_degradation,True
higher_loading_increases_risk,True
final_assay_populates_leaderboard_score,True
